In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import sqlite3
con = sqlite3.connect("../data/social.db")
tables = {"posts": "../data/social_posts.csv", "brands": "../data/brands.csv",
          "orders": "../data/gate_orders.csv", "customers": "../data/gate_customers.csv"}
for name, path in tables.items():
    pd.read_csv(path).to_sql(name, con, index=False, if_exists="replace")

toys = {
    "toy": pd.DataFrame({"name": ["a","b","c","d","e"], "team": ["x","y","x","y","x"],
                         "score": [5, 12, None, 20, 8], "city": ["UK","US","UK",None,"US"]}),
    "teams": pd.DataFrame({"team": ["x","y","z"], "coach": ["Kim","Lee","Ray"]}),
    "sponsors": pd.DataFrame({"team": ["x","x","y"], "sponsor": ["Nike","Puma","Adidas"]}),
    "staff": pd.DataFrame({"emp_id": [1,2,3,4,5], "name": ["Ana","Ben","Cat","Dev","Eli"],
                           "manager_id": [None, 1, 1, 2, 9]}),
}
for name, df in toys.items():
    df.to_sql(name, con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

SELECT p.post_id, p.brand, b.category
FROM posts p
JOIN brands b ON p.brand = b.brand

-- posts whose brand is missing from brands
SELECT p.*
FROM posts p
LEFT JOIN brands b ON p.brand = b.brand
WHERE b.brand IS NULL

SELECT p.followers AS post_followers, b.followers AS brand_followers

SELECT b.brand, COUNT(*) AS wrong, COUNT(p.post_id) AS right
FROM brands b LEFT JOIN posts p ON b.brand = p.brand
GROUP BY b.brand

-- A: filter inside ON → unmatched brands survive with 0
FROM brands b LEFT JOIN posts p ON b.brand = p.brand AND p.platform = 'TikTok'

-- B: filter in WHERE → turns your LEFT JOIN back into an INNER JOIN
FROM brands b LEFT JOIN posts p ON b.brand = p.brand
WHERE p.platform = 'TikTok'

The difference comes from the running order: FROM/JOIN happens first, and WHERE happens after.

In B, PetPal’s row has p.platform = NULL. NULL = 'TikTok' isn’t true, so WHERE drops the row.
Rule: if a filter on the right table has to keep the unmatched rows, put it in ON.
Filters on the left table can go in WHERE as normal.

SELECT e.name, m.name AS manager
FROM staff e
LEFT JOIN staff m ON e.manager_id = m.emp_id

FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT.

SELECT b.category, COUNT(*) AS n, ROUND(AVG(p.likes), 1) AS avg_likes
FROM posts p
JOIN brands b ON p.brand = b.brand
GROUP BY b.category
ORDER BY avg_likes DESC

CAST(REPLACE(o.unit_price, '$', '') AS REAL)

REPLACE works like .str.replace, and CAST(... AS REAL) works like astype(float).

In [ ]:
q(" SELECT * FROM toy")
q(" SELECT * FROM teams")
q(" SELECT * FROM sponsors")
q(" SELECT * FROM staff")

In [ ]:
q("SELECT t.name, t.team, tm.coach FROM toy t JOIN teams tm ON t.team = tm.team")
#5 rows, no z

In [ ]:
q("SELECT tm.team, tm.coach, t.name FROM teams tm LEFT JOIN toy t ON tm.team = t.team")
#6 rows, z name is null

In [ ]:
q("SELECT t.name, s.sponsor FROM toy t JOIN sponsors s ON t.team = s.team")
#8 rows, 2 times- to match the 2 sponsors Nike and Puma

In [ ]:
q("""SELECT tm.team, COUNT(*) AS n_star, COUNT(t.name) AS n_name
FROM teams tm LEFT JOIN toy t ON tm.team = t.team
GROUP BY tm.team""")
#1, n_name=0

In [ ]:
q("SELECT e.name, m.name AS manager FROM staff e JOIN staff m ON e.manager_id = m.emp_id")
#(a) How many rows?
#3
# (b) Which two employees are missing, and why is each one missing? (The reasons are different.)
# Ana, Eli-- Ana has no manager id (nan) and 9.0 has no defined employee name so Eli gets no manager identification--inner join only keeps intersecting ids
# (c) What changes if you use LEFT JOIN?
# It keeps all records from employee ids, but undefined managers become nan

In [ ]:
q("""SELECT tm.team, t.name FROM teams tm LEFT JOIN toy t ON tm.team = t.team AND t.score > 6""")
# ON version (4 rows): nothing is dropped after the join. t.score > 6 is part of the matching rule, so a player only matches their team if their score is over 6. Players a and c don’t match, so they never join. Team z has no match at all, but a LEFT JOIN keeps every left row anyway, so z gets NULL.

In [ ]:
q("""SELECT tm.team, t.name FROM teams tm LEFT JOIN toy t ON tm.team = t.team WHERE t.score > 6""")
# WHERE version (3 rows): nothing is dropped before the join. WHERE runs after the join (FROM → JOIN → WHERE). The join first builds all 6 rows, including z with NULLs. Then WHERE checks score > 6 on each row. For z that’s NULL > 6, which isn’t true, so z is removed.

ON decides what counts as a match, and WHERE filters the finished joined table

JOINS
FROM a x JOIN b y ON x.key = y.key          inner: matches only
FROM a x LEFT JOIN b y ON x.key = y.key     all of a; no match → y.* NULL
Anti-join:  LEFT JOIN ... WHERE y.key IS NULL
Self-join:  FROM staff e LEFT JOIN staff m ON e.manager_id = m.emp_id

RULES
- Prefix every column after a join (p.brand); rename clashes with AS
- Check COUNT(*) of each table and of the join, every time
- Duplicate keys multiply: 3 × 2 = 6 rows → inflated SUM/COUNT
- After LEFT JOIN, count a right-table column: COUNT(y.id), not COUNT(*)
- Filter on the RIGHT table of a LEFT JOIN → put it in ON, not WHERE
- Order: FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
- Text → number: CAST(REPLACE(col, '$', '') AS REAL)

In [ ]:
# B1. Posts with brand info

# (a) Count the rows of posts and brands.
q("SELECT COUNT(*) FROM posts")
q("SELECT COUNT(*) FROM brands")
# (b) INNER JOIN posts to brands on brand. Count the rows.
q("""SELECT COUNT(*)
FROM posts p
JOIN brands b 
ON p.brand=b.brand """)
# (c) LEFT JOIN them instead. Count the rows.
q("""SELECT COUNT(*)
FROM posts p
LEFT JOIN brands b 
ON p.brand=b.brand """)
# (d) In a Markdown cell, explain the gap between (b) and (c) with a number.
# Brands had HomeNest missing-- so 66 rows dropped for both in INNER join and only intersecting data kept
# Left join ensures all dtaa from the left posts table is kept so, it retains the 300 rows and makes missing HomeNest data from brands null. 

In [ ]:
# B2. Missing brand info

# (a) Use a LEFT JOIN + IS NULL to list each brand in posts that has no row in brands, with its post count.
q("""SELECT p.brand,  COUNT(p.post_id) 
FROM posts p
LEFT JOIN brands b ON p.brand=b.brand
WHERE b.brand IS NULL
GROUP BY p.brand""")

# (b) Does that count match the gap from B1(d)?
#Yes

In [ ]:
# B3. Brands with no posts

# (a) Start from brands and find the brand(s) with zero posts, using the anti-join pattern.
q("""SELECT b.brand
FROM brands b
LEFT JOIN posts p ON b.brand=p.brand
WHERE p.brand IS NULL
GROUP BY b.brand""")

In [ ]:
# B4. Two followers columns

# (a) Show post_id, brand, the post’s followers and the brand’s followers for the first 5 joined rows.
q("""SELECT p.post_id, p.brand, p.followers AS post_followers, b.followers AS brand_followers
FROM posts p
LEFT JOIN brands b
ON p.brand=b.brand
LIMIT 5""")
# (b) Name the columns post_followers and brand_followers.

In [ ]:
# B5. Category performance

# (a) For each brand category, show the number of posts and the average likes (rounded to 1 dp).
q("""SELECT b.category, COUNT(*) AS posts_count, ROUND(AVG(p.likes),1) AS avg_likes
FROM posts p
JOIN brands b 
ON p.brand=b.brand
GROUP BY b.category
ORDER BY avg_likes DESC""")
# (b) Sort by average likes, highest first.
# (c) Which category is top?
# food

In [ ]:
# B6. Orders ↔ customers

# (a) Count the rows of the INNER join and of the LEFT join (orders on the left).
q("""SELECT COUNT(*) 
FROM orders o
JOIN customers c
ON o.customer_id=c.customer_id""")
q("""SELECT COUNT(*) 
FROM orders o
LEFT JOIN customers c
ON o.customer_id=c.customer_id""")
# (b) List the customer_ids in orders that have no customer record, with an order count for each.
q("""SELECT COUNT(*), o.customer_id
FROM orders o
LEFT JOIN customers c 
ON o.customer_id=c.customer_id
WHERE c.customer_id IS NULL
GROUP BY o.customer_id""")
# (c) List customers with no orders.
q("""SELECT c.customer_id
FROM customers c
LEFT JOIN orders o 
ON o.customer_id=c.customer_id
WHERE o.customer_id IS NULL""")
# (d) Write one sentence on whether these match what you found in the Phase 2 gate (T7).
#yes, they match

In [ ]:
# B7. Tier analysis

# (a) Count orders per tier (inner join).
q("""SELECT tier, COUNT(*) AS tier_count
FROM orders o
JOIN customers c
ON o.customer_id=c.customer_id
GROUP BY c.tier""")
# (b) Work out the return rate % per tier (1 dp), using conditional aggregation from 3.2.
q("""SELECT 
c.tier, 
COUNT(DISTINCT c.customer_id) AS customer_count, 
COUNT(o.order_id) AS total_orders, 
SUM(CASE WHEN o.status = 'returned' THEN 1.0 ELSE 0.0 END) AS returned_orders, 
ROUND(
(SUM(CASE WHEN o.status = 'returned' THEN 1.0 ELSE 0.0 END) / COUNT(o.order_id)) * 100.0, 
1
) AS return_rate
FROM orders o 
JOIN customers c ON o.customer_id = c.customer_id 
GROUP BY c.tier;""")
# (c) Work out delivered revenue per tier using quantity * CAST(REPLACE(unit_price,'$','') AS REAL), rounded to 2 dp.
q("""SELECT
c.tier,
ROUND(SUM(o.quantity * CAST(REPLACE(o.unit_price, '$', '') AS REAL)), 2) AS revenue_per_tier
FROM orders o
LEFT JOIN customers c
ON c.customer_id = o.customer_id
WHERE o.status = 'delivered'
GROUP BY c.tier;""")
q("""
SELECT
ROUND(SUM(o.quantity * CAST(REPLACE(o.unit_price, '$', '') AS REAL)), 2) AS unmatched_revenue
FROM orders o
LEFT JOIN customers c
ON c.customer_id = o.customer_id
WHERE c.customer_id IS NULL
AND o.status = 'delivered';""")
# (d) Sanity check: sum the three tier revenues. Then find the delivered revenue of the unmatched orders. Do the two add up to the Phase 2 gate’s delivered total? Write what you find.
#the nan total matches the revenue of unmatched data

In [ ]:
# B8. TikTok posts per brand, including zeros

# (a) Starting from brands, count TikTok posts per brand, with the filter in WHERE.
# (b) Do it again with the filter in ON.
# (c) Which version shows PetPal, and with what number?
# (d) Explain the difference in one sentence.
#PetPal’s joined row has platform = NULL, so WHERE p.platform = 'TikTok' removes it.
q("""
SELECT 
b.brand, 
COUNT(p.post_id)
FROM brands b
LEFT JOIN posts p 
ON b.brand = p.brand
WHERE p.platform = 'TikTok'
GROUP BY b.brand;

""")

q("""
SELECT 
b.brand, 
COUNT(p.post_id)
FROM brands b
LEFT JOIN posts p 
ON b.brand = p.brand 
AND p.platform = 'TikTok'
GROUP BY b.brand;
""")

In [ ]:
# B9. Self-join
q("""SELECT * FROM staff
""")
# (a) List every employee with their manager’s name, keeping everyone.
q("""SELECT e.name, m.name AS manager
FROM staff e
LEFT JOIN staff m 
ON e.manager_id = m.emp_id""")
# (b) Write a Markdown note on why Eli’s manager is NULL even though his manager_id isn’t NULL.
#manager id not in employee data

In [ ]:
# B10. Interview style

# (a) For each customer country, show the number of distinct customers who ordered and the number of orders.
q("""SELECT
c.country,
COUNT(DISTINCT o.customer_id) AS customers_ordered,
COUNT(o.order_id) AS total_orders
FROM customers c
LEFT JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY total_orders DESC""")

# (b) Sort by orders, highest first.
# (c) Work out orders per customer for the top country (by hand is fine).
q("""
SELECT
    COUNT(DISTINCT o.customer_id) AS customers_unique,
    COUNT(o.order_id) AS total_orders,
    COUNT(o.order_id) * 1.0 / COUNT(DISTINCT o.customer_id) AS orders_per_customer
FROM customers c
LEFT JOIN orders o
    ON o.customer_id = c.customer_id
WHERE c.country = 'UK';
""")
# q("""
# SELECT c.customer_id AS unique_customers,
# COUNT(o.order_id) AS orders_count
# FROM customers c
# LEFT JOIN orders o
# ON o.customer_id = c.customer_id
# WHERE c.country = 'UK'
# GROUP BY c.customer_id;
# """)

In [ ]:
q("""SELECT p.brand, p.likes, b.category 
FROM posts p 
JOIN brands b 
ON p.brand = b.brand""")

In [ ]:
q("""
SELECT b.brand, COUNT(p.brand) AS n_posts
FROM brands b 
LEFT JOIN posts p ON b.brand = p.brand
GROUP BY b.brand
""")
# COUNT(*) counts everything

In [ ]:
q("""SELECT COUNT(*), p.platform
FROM posts p 
JOIN brands b 
ON p.brand = b.brand
GROUP BY p.platform""")
# platforms not grouped

In [ ]:
q("""SELECT team, SUM(score) AS total
FROM toy
GROUP BY team""")
#the join causes the rows to come 2 times, doubling the results

In [ ]:
# Level 1: INNER JOIN

# P1. Using toy and teams:
q("""SELECT * FROM toy
""")
q("""SELECT * FROM teams""")
# (a) List each player’s name and their coach.
q("""
SELECT p.name, c.coach
FROM toy p
LEFT JOIN teams c
ON p.team=c.team
WHERE score>6
""")
# (b) Now show only players with a score over 6.

In [ ]:
# P2.

# (a) Count players per coach.
# (b) Which coach has the most players?
q("""
SELECT c.coach, COUNT(p.name) AS player_count
FROM toy p
LEFT JOIN teams c
ON p.team=c.team
GROUP BY c.coach
ORDER BY player_count DESC
LIMIT 1""")

In [ ]:
# Level 2: LEFT JOIN and NULL

# P3.

# (a) List every team with its coach and number of players. Team z must show 0.
q("""
SELECT c.team, c.coach, COUNT(p.name) AS player_count
FROM teams c
LEFT JOIN toy p
ON p.team = c.team
GROUP BY c.team
""")

# (b) Which COUNT did you use, and why?
# COUNT(*) would count z’s NULL row as 1.

# P4.

# (a) Using teams and sponsors, find the team(s) with no sponsor (anti-join).
q("""
SELECT * FROM teams""")
q("""
SELECT * FROM sponsors""")

q("""
SELECT t.team
FROM teams t
LEFT JOIN sponsors s
ON t.team = s.team
WHERE s.sponsor IS NULL""")
# (b) Predict how many rows teams LEFT JOIN sponsors returns before you check.
#4


In [ ]:
# Level 3: Predicting row counts

# No running until you’ve written all your predictions.

# P5. Predict the row counts:

# (a) teams JOIN sponsors
q("""
SELECT *
FROM teams t
JOIN sponsors s
ON t.team = s.team""")
# (b) teams LEFT JOIN sponsors
q("""
SELECT *
FROM teams t
LEFT JOIN sponsors s
ON t.team = s.team""")
# (c) sponsors LEFT JOIN toy (careful: which table is on the left now, and how many players match each sponsor row?)
q("""
SELECT *
FROM sponsors s
LEFT JOIN toy t
ON t.team = s.team""")

# (d) Now run all three and explain any you got wrong, using the 0 / 1 / many table above.

# P6. Three tables. You can chain joins: FROM a JOIN b ON … JOIN c ON ….

# (a) List every player with their coach and sponsor (toy + teams + sponsors). Predict the row count first.
q("""SELECT p.name, c.coach, s.sponsor
FROM toy p
LEFT JOIN teams c
ON p.team=c.team
JOIN sponsors s
ON c.team=s.team""")
# (b) Start from teams instead and use LEFT JOINs for both, so team z appears. Predict the row count.
q("""SELECT p.name, c.coach, s.sponsor
FROM teams c
LEFT JOIN toy p
ON p.team=c.team
LEFT JOIN sponsors s
ON c.team=s.team""")

In [ ]:
# Level 4: Real data, join + group

# P7. posts ⋈ brands:

# (a) For each brand hq_country, show the number of posts and the total likes.
q("""SELECT b.hq_country, COUNT(p.post_id) AS post_count, SUM(p.likes) AS total_likes
FROM brands b
JOIN posts p
ON b.brand=p.brand
GROUP BY b.hq_country""")

# (b) Why is HomeNest not counted? Answer in one line.
#was not in brands and we joined brands to posts

In [ ]:
# P8. orders ⋈ customers, delivered orders only:

# (a) Count delivered orders per country, highest first.
q("""SELECT c.country, COUNT(o.order_id) AS order_count
FROM orders o
JOIN customers c
ON c.customer_id=o.customer_id
AND o.status='delivered'
GROUP BY c.country
ORDER BY order_count DESC""")
# (b) Show the average rating per tier (2 dp).
q("""SELECT c.tier, ROUND(AVG(o.rating),2) AS avg_rating
FROM orders o
JOIN customers c
ON c.customer_id=o.customer_id
GROUP BY c.tier
ORDER BY avg_rating DESC""")
# (c) Which tier rates highest?
#Plus

In [ ]:
# P9. Likes per £ of budget:

# (a) For every brand, show monthly_budget, total likes, and likes per £ (total likes ÷ budget, 2 dp, using 1.0). Sort highest first.
q("""SELECT b.brand, b.monthly_budget, SUM(p.likes) AS total_likes, ROUND((SUM(p.likes)*1.0/b.monthly_budget*1.0),2) AS likes_per_pound
FROM brands b
LEFT JOIN posts p
ON p.brand=b.brand
GROUP BY b.brand
ORDER BY likes_per_pound DESC""")
# (b) What does PetPal show, and why?
# PetPal has no posts, so SUM(likes) is NULL, and NULL ÷ 3000 is still NULL.
# (c) Which brand gets the most likes per £?
# Glowly

In [ ]:
# P10. Heavy buyers:

# (a) List customers (id and country) with 8 or more orders, using JOIN + GROUP BY + HAVING.
q("""SELECT c.customer_id, c.country, COUNT(o.order_id) AS order_count
FROM customers c
JOIN orders o
ON c.customer_id=o.customer_id
GROUP BY c.customer_id, c.country
HAVING COUNT(o.order_id)>=8""")
# (b) How many are there?
#5

In [ ]:
# Level 5: Harder

# P11. Filtering in ON:

# (a) For each tier, count cancelled orders, starting from customers. Put the status filter in ON.
q("""SELECT c.tier, COUNT(o.order_id)
FROM customers c
LEFT JOIN orders o
ON c.customer_id=o.customer_id
AND o.status='cancelled'
GROUP BY c.tier""")
# (b) How many customers have never had a cancelled order? Use an anti-join, with the status filter in ON.

q("""
SELECT COUNT(c.customer_id)
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'cancelled'
WHERE o.order_id IS NULL
""")

# (c) Run (b) again with the status filter in WHERE instead. Explain the result.

q("""
SELECT COUNT(c.customer_id)
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.status = 'cancelled'
  AND o.order_id IS NULL
""")
# q("""
# SELECT
#     c.customer_id,
#     o.customer_id,
#     o.order_id,
#     o.status
# FROM customers c
# LEFT JOIN orders o
#     ON c.customer_id = o.customer_id
#     AND o.status = 'cancelled'
#     WHERE o.order_id IS NULL
#     ;""")

In [ ]:
# P12. Self-join:

# (a) For every employee in staff, count how many people report to them (their direct reports). Zeros must show. Hint: the left table is the managers’ copy.
q("""SELECT * FROM staff""")

q("""SELECT e.name, COUNT(m.manager_id) AS reporting_employees
FROM staff e
LEFT JOIN staff m
ON m.manager_id=e.emp_id
GROUP BY e.name""")
# (b) Find employees whose manager_id points to someone who doesn’t exist (a self anti-join). Make sure Ana isn’t in your answer.
q("""
SELECT e.name
FROM staff e
LEFT JOIN staff m
    ON e.manager_id = m.emp_id
WHERE e.manager_id IS NOT NULL
AND m.emp_id IS NULL
""")

In [ ]:
q("""SELECT category, SUM(monthly_budget) AS total_budget
  FROM brands 
  GROUP BY category""")
#pets disappeared because it wasnt in posts and got removed in the join

In [ ]:
# P14. Interview style:

# (a) How many distinct customers per tier have made at least one return?
q("""
SELECT
    c.tier,
    COUNT(DISTINCT c.customer_id) AS customers_returned
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'returned'
GROUP BY c.tier
""")
# (b) How many customers have never returned anything?

q("""
SELECT
    COUNT(DISTINCT c.customer_id) AS customers_never_returned
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'returned'
WHERE o.order_id IS NULL
""")

# (c) Do (a) and (b) together add up to the number of customers? Show the sum.
q(""" SELECT COUNT(DISTINCT customer_id) FROM customers""") #= 36+84